## 0. 환경 (uv + Intel XPU)

터미널에서 실행 (GigaTime 루트, PowerShell):
```powershell
uv venv --python 3.11
uv pip install --python .venv/Scripts/python.exe --extra-index-url https://download.pytorch.org/whl/xpu "torch==2.13.0+xpu" pandas numpy scikit-learn matplotlib jupyter ipykernel
.venv\Scripts\python.exe -m ipykernel install --user --name gigatime-lstm-xpu --display-name "Python 3.11 (gigatime-lstm-xpu)"
```
노트북 커널은 `Python 3.11 (gigatime-lstm-xpu)` 선택. 아래 0-1 셀은 부트스트랩/검증용.

In [ ]:
# Cell 0 (B안): torch 2.13.0+xpu + XPU 사용 가능하면 설치 스킵
# 주의: torch가 이미 import된 live 커널에서 재설치하면 로드된 DLL이 교체되어
# 네이티브 크래시(ExitCode 3221226505)로 커널이 죽는다. 재설치는 터미널에서만 할 것.
import sys, subprocess, shutil
print(sys.executable)
def _need_install():
    try:
        import torch
        xpu_ok = hasattr(torch, "xpu") and torch.xpu.is_available()
        print("torch:", torch.__version__, "xpu:", xpu_ok)
        return not (torch.__version__.startswith("2.13.0+xpu") and xpu_ok)
    except ImportError:
        print("torch not installed")
        return True
if _need_install():
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("uv 바이너리를 찾을 수 없음. 터미널에서 uv 설치 후 재실행")
    print("uv:", uv)
    subprocess.run([uv, "--version"], check=True)
    subprocess.run([
        uv, "pip", "install", "--python", sys.executable,
        "--extra-index-url", "https://download.pytorch.org/whl/xpu",
        "torch==2.13.0+xpu",
        "pandas", "numpy", "scikit-learn", "matplotlib", "jupyter", "ipykernel"
    ], check=True)
    print("installed. 커널 재시작 후 다음 셀 실행")
else:
    print("skip install: XPU torch OK. 다음 셀로 이동")

In [ ]:
# Cell 1: 환경 검증 (기대: 2.13.0+xpu / True / Intel Arc 140V)
import sys, torch
print(sys.executable)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("xpu:", torch.xpu.is_available() if hasattr(torch, "xpu") else False)
if hasattr(torch, "xpu") and torch.xpu.is_available():
    print(torch.xpu.get_device_name(0))

# 공장 전력사용량 1시간 후 예측 - LSTM (baseline)

입력: 과거 24시간의 temperature / humidity / energy_relative_pct
출력: 다음 1시간의 target_energy_relative_pct (변화율)

이 노트북은 baseline 체크포인트(factory_energy_lstm.pt) 생성용이다.  
`target_time_utc` 기준 시간순으로 미리 분리된 data/train.csv, data/valid.csv, data/test.csv를 사용합니다. Validation은 epoch마다 평가하고 Early Stopping에 사용하며 Test는 최종 모델 확정 후 한 번만 평가합니다.

In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    torch.xpu.manual_seed_all(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = torch.device("xpu")
else:
    DEVICE = torch.device("cpu")
print("DEVICE:", DEVICE)

In [ ]:
# 분할 파일 직접 사용 (train/valid/test 이미 분리됨)
TRAIN_PATH = "data/train.csv"
VALID_PATH = "data/valid.csv"
TEST_PATH = "data/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)
test_df = pd.read_csv(TEST_PATH)
for _df in (train_df, valid_df, test_df):
    _df["target_time_utc"] = pd.to_datetime(_df["target_time_utc"], utc=True)
    _df.sort_values("target_time_utc", inplace=True)
    _df.reset_index(drop=True, inplace=True)
print(train_df.shape, valid_df.shape, test_df.shape)
print(train_df[["target_time_utc", "target_energy_relative_pct"]].head())

## 1. 24시간 × 3개 feature 구성

각 행은 `lag_24h -> ... -> lag_1h`의 24시간 window이며, timestep마다 `[temp, humidity, energy]` 3개 feature를 갖습니다.

In [ ]:
features = []
for lag in range(24, 0, -1):
    features += [f"temp_F_lag_{lag}h", f"humi_pct_lag_{lag}h", f"energy_relative_pct_lag_{lag}h"]
target_col = "target_energy_relative_pct"
for name, _df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in features + [target_col] if c not in _df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing columns: {missing}")
def make_xy(_df):
    X_flat = _df[features].to_numpy(dtype=np.float32)
    y = _df[target_col].to_numpy(dtype=np.float32)
    ts = _df["target_time_utc"].to_numpy()
    return X_flat.reshape(-1, 24, 3), y, ts
X_train, y_train, ts_train = make_xy(train_df)
X_valid, y_valid, ts_valid = make_xy(valid_df)
X_test, y_test, ts_test = make_xy(test_df)
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

## 2. 시간순 분할 (train/valid/test 파일 그대로 사용)

파일이 이미 시간순으로 분리되어 있어 별도 split 없음. 순서·누락만 검증합니다.

In [ ]:
print("Train:", X_train.shape, ts_train[0], "->", ts_train[-1])
print("Valid:", X_valid.shape, ts_valid[0], "->", ts_valid[-1])
print("Test :", X_test.shape, ts_test[0], "->", ts_test[-1])
assert ts_train.max() <= ts_valid.min(), "train/valid 시간 겹침"
assert ts_valid.max() <= ts_test.min(), "valid/test 시간 겹침"

## 3. Scaling

Scaler는 Train에만 fit합니다. Validation/Test는 transform만 합니다.

In [ ]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()
feature_scaler.fit(X_train.reshape(-1, 3))
target_scaler.fit(y_train.reshape(-1, 1))

def scale_X(a):
    shape = a.shape
    return feature_scaler.transform(a.reshape(-1, 3)).reshape(shape).astype(np.float32)

X_train_s, X_valid_s, X_test_s = map(scale_X, [X_train, X_valid, X_test])
y_train_s = target_scaler.transform(y_train.reshape(-1, 1)).astype(np.float32).ravel()
y_valid_s = target_scaler.transform(y_valid.reshape(-1, 1)).astype(np.float32).ravel()
y_test_s = target_scaler.transform(y_test.reshape(-1, 1)).astype(np.float32).ravel()

In [ ]:
BATCH_SIZE = 64  # Arc 140V 공유메모리 OOM 방지: 256 -> 64
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_s), torch.from_numpy(y_train_s)), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
valid_loader = DataLoader(TensorDataset(torch.from_numpy(X_valid_s), torch.from_numpy(y_valid_s)), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test_s), torch.from_numpy(y_test_s)), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

## 4. LSTM 모델

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

model = LSTMRegressor().to(DEVICE)
criterion = nn.MSELoss()
# XPU OOM fix: foreach Adam(_foreach_addcdiv_)이 Level Zero에서 OOM → single-tensor 경로 강제
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, foreach=False)
print(model)

## 5. Train → Validation 반복 / Early Stopping

In [ ]:
def evaluate_loader(model, loader):
    model.eval(); preds=[]; actuals=[]
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(pred); actuals.append(yb.numpy())
    pred_s=np.concatenate(preds); actual_s=np.concatenate(actuals)
    pred=target_scaler.inverse_transform(pred_s.reshape(-1,1)).ravel()
    actual=target_scaler.inverse_transform(actual_s.reshape(-1,1)).ravel()
    return np.sqrt(mean_squared_error(actual,pred)), mean_absolute_error(actual,pred), pred, actual

EPOCHS=50; PATIENCE=7
best_val_rmse=float("inf"); best_state=None; patience=0; history=[]

for epoch in range(1,EPOCHS+1):
    model.train()
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE, non_blocking=False),yb.to(DEVICE, non_blocking=False)
        optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(xb),yb)
        loss.backward(); optimizer.step()
        del loss
    if hasattr(torch, "xpu") and DEVICE.type == "xpu":
        torch.xpu.empty_cache()
    train_rmse, train_mae, _, _ = evaluate_loader(model, train_loader)
    val_rmse, val_mae, _, _ = evaluate_loader(model, valid_loader)
    if hasattr(torch, "xpu") and DEVICE.type == "xpu":
        torch.xpu.empty_cache()
    history.append([epoch,train_rmse,train_mae,val_rmse,val_mae])
    print(f"Epoch {epoch:02d} | Train RMSE {train_rmse:.4f} | Val RMSE {val_rmse:.4f} | Val MAE {val_mae:.4f}")
    if val_rmse < best_val_rmse:
        best_val_rmse=val_rmse; best_state=copy.deepcopy(model.state_dict()); patience=0
    else:
        patience += 1
        if patience >= PATIENCE:
            print("Early stopping."); break

model.load_state_dict(best_state)
history_df=pd.DataFrame(history,columns=["epoch","train_rmse","train_mae","val_rmse","val_mae"])
history_df

## 6. 최종 Test 평가

Best validation checkpoint를 확정한 뒤 Test를 평가합니다.

In [ ]:
test_rmse, test_mae, test_pred, test_actual = evaluate_loader(model, test_loader)
print("=== Final Test ===")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE : {test_mae:.4f}")

## 7. 시간에 따른 성능 변화

장기간 데이터에서 drift 여부를 살펴보기 위해 Test 결과를 연도/월별로 분해합니다.

In [ ]:
test_result=pd.DataFrame({"target_time_utc":pd.to_datetime(ts_test),"actual":test_actual,"predicted":test_pred})
test_result["year"]=test_result.target_time_utc.dt.year
test_result["month"]=test_result.target_time_utc.dt.to_period("M").astype(str)
test_result["squared_error"]=(test_result.actual-test_result.predicted)**2
test_result["absolute_error"]=abs(test_result.actual-test_result.predicted)
yearly=test_result.groupby("year").agg(RMSE=("squared_error",lambda x:np.sqrt(x.mean())),MAE=("absolute_error","mean"),N=("absolute_error","size")).reset_index()
monthly=test_result.groupby("month").agg(RMSE=("squared_error",lambda x:np.sqrt(x.mean())),MAE=("absolute_error","mean"),N=("absolute_error","size")).reset_index()
display(yearly); display(monthly)

In [ ]:
torch.save({"model_state_dict":model.state_dict(),"feature_scaler":feature_scaler,"target_scaler":target_scaler,"input_shape":(24,3),"feature_order":features,"target":target_col},"factory_energy_lstm.pt")
print("saved: factory_energy_lstm.pt")